<a href="https://colab.research.google.com/github/nee1k/patra-toolkit/blob/main/MegaDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!git clone https://github.com/ultralytics/yolov5.git
!pip install -r /content/yolov5/requirements.txt

fatal: destination path 'yolov5' already exists and is not an empty directory.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.6/914.6 kB 50.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu

In [9]:
import sys
sys.path.append("/content/yolov5")

In [16]:
import torch
import cv2
import numpy as np

# Import YOLOv5 utilities (make sure yolov5 is on your path)
from utils.general import non_max_suppression
from utils.augmentations import letterbox
try:
    from utils.general import scale_coords  # YOLOv5 < v7
except ImportError:
    from utils.general import scale_boxes as scale_coords  # YOLOv5 v7+

from huggingface_hub import hf_hub_download

def simple_md_inference(image_path, device='cpu', conf_thres=0.25):
    """
    Loads the MegaDetector V5 checkpoint, preprocesses the image,
    runs inference, and returns a list of {label, probability} dicts.
    """
    # 1. Download checkpoint from Hugging Face
    model_path = hf_hub_download(repo_id="nkarthikeyan/MegaDetectorV5",
                                 filename="md_v5a.0.0.pt")

    # 2. Load YOLO model from checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    # The actual YOLO model is stored under 'model'
    yolov5_model = checkpoint['model'].float().fuse().eval()

    # 3. Read and preprocess image
    orig_img = cv2.imread(image_path)            # BGR (OpenCV)
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)  # Convert to RGB
    # Letterbox to a fixed size (e.g., 640)
    resized_img = letterbox(orig_img, new_shape=640, stride=64, auto=True)[0]

    # Convert HWC -> CHW, float [0,1]
    resized_img = resized_img.transpose(2, 0, 1)
    resized_img = np.ascontiguousarray(resized_img, dtype=np.float32) / 255.0

    # Create a batch of size 1
    tensor_img = torch.from_numpy(resized_img).unsqueeze(0).to(device)

    # 4. Inference
    with torch.no_grad():
        prediction = yolov5_model(tensor_img)[0]

    # 5. Non-max suppression
    detections = non_max_suppression(prediction, conf_thres=conf_thres, iou_thres=0.45)

    # 6. Map detections to label/conf
    results = []
    if len(detections) > 0 and detections[0] is not None:
        # Rescale boxes back to original image size
        det = detections[0]
        det[:, :4] = scale_coords(resized_img.shape[1:], det[:, :4], orig_img.shape).round()

        # YOLO detection format: x1, y1, x2, y2, conf, class
        for *xyxy, conf, cls in det.tolist():
            results.append({
                "label": int(cls),          # YOLO class index
                "probability": float(conf)  # confidence score
            })

    return results

# --- Example usage ---
if __name__ == "__main__":
    image_path = "megadetector_test_img.JPG"
    detections = simple_md_inference(image_path, device='cpu', conf_thres=0.25)
    print("Detections:", detections)


Fusing layers... 
Model summary: 574 layers, 139990096 parameters, 0 gradients, 207.9 GFLOPs


Detections: [{'label': 0, 'probability': 0.9130892157554626}]


## 4. Model Card Generation with Patra Toolkit

Create a comprehensive Model Card to document the model's details, including metadata, bias analysis, and explainability metrics.


In [ ]:
import os
import torch
from PIL import Image
from huggingface_hub import hf_hub_download

# If your PTDetector code is in a separate file, import it:
# from your_module import PTDetector

###############################################################################
# 1. Download the MegaDetectorV5 checkpoint from Hugging Face
###############################################################################
model_file = hf_hub_download(repo_id="nkarthikeyan/MegaDetectorV5", filename="md_v5a.0.0.pt")
print("Downloaded model file:", model_file)

###############################################################################
# 2. Instantiate PTDetector with the downloaded checkpoint
#    (set force_cpu=True if you want to use CPU only)
###############################################################################
detector = PTDetector(model_path=model_file, force_cpu=True)
print("PTDetector initialized.")

###############################################################################
# 3. Run inference on a test image
###############################################################################
# Make sure you have an image at this path. Alternatively, pick any local path.
test_image_path = "/content/megadetector_test_img.JPG"

# Load the image as a PIL Image
img = Image.open(test_image_path).convert("RGB")

# Set a detection threshold (e.g., 0.2). Adjust as needed for your scenario.
detection_threshold = 0.2

# Run inference
results = detector.generate_detections_one_image(
    img_original=img,
    image_id=os.path.basename(test_image_path),
    detection_threshold=detection_threshold
)

###############################################################################
# 4. Print and/or visualize the results
###############################################################################
print("Inference results:", results)

# Optionally, if you want to save detection visualization:
# (Assuming the function single_image_detection() in PTDetector is slightly different
#  from your code, you can do it directly, or you might do your own bounding box drawing)
#
# Using your approach with pw_utils:
#
# from PytorchWildlife import utils as pw_utils
# pw_utils.save_detection_images(results, "./demo_output", overwrite=True)
# print("Inference completed and detection images saved.")


{
    "name": "Image Recognition Model using Hugging Face",
    "version": "0.1",
    "short_description": "Image recognition model for demonstration of Patra Model Cards using Hugging Face.",
    "full_description": "We have trained a deep learning model using the Hugging Face framework for image classification tasks. We use this data to run Patra model cards to capture metadata about the model.",
    "keywords": "image recognition, hugging face, patra",
    "author": "Neelesh Karthikeyan",
    "input_type": "Image",
    "category": "classification",
    "input_data": "https://huggingface.co/datasets/cifar10",
    "output_data": "https://huggingface.co/models/sachith/image_recognition_model_v01",
    "foundational_model": "None",
    "ai_model": {
        "name": "Image Recognition Hugging Face Model",
        "version": "0.1",
        "description": "Image classification model using Hugging Face Transformers and CNNs for various image recognition tasks.",
        "owner": "Neelesh Ka

## 5. Validation and Submission

After generating the Model Card, it's crucial to validate and ensure that all information is accurate and properly formatted. Submitting the Model Card allows it to be integrated into other systems or platforms as needed.

In [ ]:
# Validate the Model Card
mc.validate()

# Submit the Model Card
# mc.submit("<patra_server_url>")

# Save the Model Card to a JSON file
mc.save("huggingface_modelcard.json")
